<a href="https://colab.research.google.com/github/Amulya-Rao-mj/GENAI-lab/blob/main/DocuMind_PDF_Chatbot_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%%capture
!pip install -q transformers accelerate sentence-transformers faiss-cpu pypdf gradio langchain-text-splitters pytesseract pdf2image
!apt-get -qq update && apt-get -qq install -y tesseract-ocr poppler-utils

In [4]:
import os, re, inspect, threading
import numpy as np
import torch
import faiss
import gradio as gr
from pypdf import PdfReader
from pdf2image import convert_from_path
import pytesseract
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Settings you can tweak ----
LLM_NAME       = "Qwen/Qwen2.5-1.5B-Instruct"
EMBED_NAME     = "BAAI/bge-small-en-v1.5"
CHUNK_SIZE     = 700
CHUNK_OVERLAP  = 120
TOP_K          = 4
MAX_NEW_TOKENS = 300
MEMORY_TURNS   = 3
print("Running on:", DEVICE)

Running on: cuda


In [5]:
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype="auto", device_map="auto")
embedder = SentenceTransformer(EMBED_NAME, device=DEVICE)
print("Models ready ✔")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

... (13 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Models ready ✔


In [6]:
def extract_pages(path):
    """Return a list of (page_number, text). Falls back to OCR for scanned PDFs."""
    reader = PdfReader(path)
    pages = [(i + 1, (p.extract_text() or "").strip()) for i, p in enumerate(reader.pages)]
    pages = [(n, t) for n, t in pages if len(t) > 20]

    if pages:
        return pages, "text layer"

    images = convert_from_path(path, dpi=200)
    pages = []
    for i, img in enumerate(images):
        t = pytesseract.image_to_string(img).strip()
        if len(t) > 20:
            pages.append((i + 1, t))
    return pages, "OCR"


splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

def make_chunks(pages):
    out = []
    for page_no, text in pages:
        text = re.sub(r"\s+", " ", text)
        for piece in splitter.split_text(text):
            out.append({"page": page_no, "text": piece})
    return out

In [7]:
class DocStore:
    def __init__(self):
        self.chunks, self.index, self.name = [], None, None

    def build(self, chunks, name):
        vecs = embedder.encode(
            [c["text"] for c in chunks],
            normalize_embeddings=True, convert_to_numpy=True, batch_size=32
        ).astype("float32")
        self.index = faiss.IndexFlatIP(vecs.shape[1])   # inner product == cosine when normalised
        self.index.add(vecs)
        self.chunks, self.name = chunks, name

    def search(self, query, k=TOP_K):
        q = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        scores, ids = self.index.search(q, min(k, len(self.chunks)))
        return [dict(self.chunks[i], score=float(s)) for s, i in zip(scores[0], ids[0]) if i != -1]

store = DocStore()

In [8]:
SYSTEM_PROMPT = (
    "You are DocuMind, a precise assistant that answers questions about a single PDF. "
    "Use ONLY the provided excerpts. If the answer is not in them, reply exactly: "
    "'Sorry, I couldn't find that in the document.' Keep answers clear and concise, "
    "and mention page numbers when helpful."
)

def build_messages(question, hits, history):
    context = "\n\n".join(f"(Page {h['page']}) {h['text']}" for h in hits)
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    for m in history[-2 * MEMORY_TURNS:]:
        content = m["content"].split("\n\n---")[0]      # strip the sources footer
        msgs.append({"role": m["role"], "content": content})
    msgs.append({"role": "user", "content": f"EXCERPTS:\n{context}\n\nQUESTION: {question}"})
    return msgs


def stream_answer(question, hits, history):
    text = tokenizer.apply_chat_template(
        build_messages(question, hits, history), tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(llm.device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    kwargs = dict(**inputs, streamer=streamer, max_new_tokens=MAX_NEW_TOKENS,
                  do_sample=True, temperature=0.2, top_p=0.9, repetition_penalty=1.05)
    threading.Thread(target=llm.generate, kwargs=kwargs).start()

    partial = ""
    for piece in streamer:
        partial += piece
        yield partial

In [9]:
CSS = """
.gradio-container {max-width: 1100px !important; margin: auto;}
#hero {text-align:center; padding: 22px 10px; border-radius: 18px; color: white;
       background: linear-gradient(135deg,#4f46e5 0%,#7c3aed 55%,#db2777 100%);}
#hero h1 {margin: 0; font-size: 2.1rem;}
#hero p {margin: 6px 0 0; opacity: .9;}
#status {border-left: 4px solid #7c3aed; padding: 8px 12px; border-radius: 6px;}
#sources {font-size: .85rem;}
footer {display:none !important;}
"""

THEME = gr.themes.Soft(primary_hue="violet", secondary_hue="pink", neutral_hue="slate")

EXAMPLES = ["Give me a short summary of this document.",
            "What are the key points?",
            "List any important dates, numbers or names."]


def load_pdf(file):
    if file is None:
        return "⚠️ Please choose a PDF first.", gr.update(interactive=False), [], ""
    path = file if isinstance(file, str) else file.name
    try:
        pages, method = extract_pages(path)
        if not pages:
            return "❌ Couldn't read any text from this PDF.", gr.update(interactive=False), [], ""
        chunks = make_chunks(pages)
        store.build(chunks, os.path.basename(path))
    except Exception as e:
        return f"❌ Error: {e}", gr.update(interactive=False), [], ""
    msg = (f"✅ **{store.name}** indexed — {len(pages)} pages, {len(chunks)} chunks "
           f"(extracted via {method}). Ask away!")
    return msg, gr.update(interactive=True), [], ""


def render_sources(hits):
    rows = []
    for h in hits:
        snippet = h["text"][:230] + ("…" if len(h["text"]) > 230 else "")
        rows.append(f"**📄 Page {h['page']}** · relevance `{h['score']:.2f}`\n> {snippet}")
    return "\n\n".join(rows)


def respond(message, history):
    history = history or []
    if not message.strip():
        yield history, "", gr.update()
        return
    if store.index is None:
        history = history + [{"role": "user", "content": message},
                             {"role": "assistant", "content": "📎 Please upload and index a PDF first."}]
        yield history, "", ""
        return

    hits = store.search(message)
    pages = ", ".join(f"p.{p}" for p in sorted({h["page"] for h in hits}))
    footer = f"\n\n---\n📌 *Sources: {pages}*"
    history = history + [{"role": "user", "content": message}]

    for partial in stream_answer(message, hits, history[:-1]):
        yield history + [{"role": "assistant", "content": partial}], "", render_sources(hits)
    yield history + [{"role": "assistant", "content": partial + footer}], "", render_sources(hits)


def make_chatbot():
    try:
        return gr.Chatbot(type="messages", height=480, label="Conversation", show_copy_button=True)
    except TypeError:              # newer Gradio versions use messages format by default
        return gr.Chatbot(height=480, label="Conversation")


blocks_kwargs, launch_kwargs = {}, {}
if "theme" in inspect.signature(gr.Blocks.launch).parameters:   # Gradio 6+
    launch_kwargs = dict(theme=THEME, css=CSS)
else:
    blocks_kwargs = dict(theme=THEME, css=CSS)

with gr.Blocks(title="DocuMind", **blocks_kwargs) as app:
    gr.HTML('<div id="hero"><h1>📚 DocuMind</h1>'
            '<p>Upload a PDF and have a conversation with it — answers are grounded in your document.</p></div>')

    with gr.Row():
        with gr.Column(scale=1, min_width=300):
            pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"])
            index_btn = gr.Button("⚡ Index document", variant="primary")
            status = gr.Markdown("Waiting for a PDF…", elem_id="status")
            with gr.Accordion("🔎 Retrieved passages", open=True):
                sources_box = gr.Markdown("*Sources for each answer appear here.*", elem_id="sources")

        with gr.Column(scale=2):
            chat = make_chatbot()
            with gr.Row():
                box = gr.Textbox(placeholder="Ask something about your document…", show_label=False,
                                 scale=5, interactive=False, container=False)
                send = gr.Button("Send ➤", variant="primary", scale=1)
            gr.Examples(EXAMPLES, inputs=box, label="Try asking")
            clear = gr.Button("🗑️ Clear chat", size="sm")

    index_btn.click(load_pdf, pdf_input, [status, box, chat, sources_box])
    box.submit(respond, [box, chat], [chat, box, sources_box])
    send.click(respond, [box, chat], [chat, box, sources_box])
    clear.click(lambda: ([], "*Sources for each answer appear here.*"), None, [chat, sources_box])

app.queue().launch(share=True, debug=True, **launch_kwargs)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a8d026af0566e8ef19.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a8d026af0566e8ef19.gradio.live
